In [19]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")          # trabalhar sempre a partir da raiz do projeto

import duckdb
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect() #data base on memory

con.sql("""
CREATE OR REPLACE VIEW survey AS
SELECT * FROM 'data/processed/survey_clean.parquet'
""")

con.sql("Describe survey").df()[["column_name", "column_type"]]

,column_name,column_type
0,response_id,BIGINT
1,employment,VARCHAR
2,country,VARCHAR
3,dev_type,VARCHAR
4,ed_level,VARCHAR
5,org_size,VARCHAR
6,remote_work,VARCHAR
7,years_pro,DOUBLE
8,years_pro_alt,VARCHAR
9,years_code,DOUBLE


In [20]:
con.sql("SELECT YEAR, COUNT(*) AS n FROM survey GROUP BY year ORDER BY year").df()

,year,n
0,2024,20669
1,2025,18513


## AI adoption per year

In [21]:
query = """
SELECT year, 
    COUNT(*) AS n_total, 
    COUNT(uses_ai) AS n_responses, 
    ROUND(100 * AVG(uses_ai::INT),1) As pct_ai_use
FROM survey
GROUP BY year
Order By Year
"""

con.sql(query).df()

,year,n_total,n_responses,pct_ai_use
0,2024,20669,20669,62.6
1,2025,18513,18037,80.8


In [22]:

d1 = con.sql(query).df()
d1["year"] = d1["year"].astype(str) 
fig = px.bar(d1, x="year", y="pct_ai_use", text="pct_ai_use",
             title="AI adoption between developers workers (2024 vs 2025)",
             labels={"pct_ai_use": "% that use ai", "year": ""})
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 100])
fig.show()

## Frequency of AI use in 2025

In [23]:
q_frequency = """
SELECT ai_use_frequency,
    COUNT(*) AS n, 
    COUNT(uses_ai) AS n_responses, 
    ROUND(100 * COUNT(*) /SUM(COUNT(*)) OVER (), 1) AS pct
FROM survey
WHERE year = 2025 AND uses_ai
GROUP BY ai_use_frequency
ORDER BY n DESC
"""

d2 = con.sql(q_frequency).df()
d2

,ai_use_frequency,n,n_responses,pct
0,daily,8721,8721,59.9
1,weekly,3386,3386,23.2
2,monthly_or_less,2464,2464,16.9


In [24]:
fig = px.bar(d2, x="pct", y="ai_use_frequency", orientation="h", text="pct",
             title="How often do developers use AI (2025, users only)",
             labels={"pct": "% of users", "ai_use_frequency": ""},
             category_orders={"ai_use_frequency": ["daily", "weekly", "monthly_or_less"]})
fig.update_traces(texttemplate="%{text}%")
fig.update_yaxes(autorange="reversed")
fig.show()

## Adoption by country and the decision on the minimum sample size.

In [25]:
query_countries ="""
SELECT country,
        Count(*) FILTER (WHERE year = 2024) AS n_2024,
        COUNT(*) FILTER (WHERE year = 2025) AS n_2025,
FROM survey
WHERE uses_ai IS NOT NULL
GROUP BY country
ORDER BY LEAST(n_2024, n_2025) DESC
LIMIT 25
"""

con.sql(query_countries).df()

,country,n_2024,n_2025
0,United States of America,4306,4220
1,Germany,1918,1749
2,United Kingdom of Great Britain and Northern I...,1260,1223
3,India,923,802
4,France,830,781
5,Canada,790,721
6,Netherlands,502,500
7,Brazil,615,456
8,Italy,481,444
9,Spain,509,438


In [26]:
q_limiares = """
WITH por_pais AS(
    SELECT country,
         COUNT(*) FILTER (WHERE year = 2024) AS n_2024,
         COUNT(*) FILTER (WHERE year = 2025) AS n_2025
  FROM survey
  WHERE uses_ai IS NOT NULL
  GROUP BY country
)
SELECT t.limiar,
        COUNT(*) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar) AS countries_who_pass
FROM por_pais
CROSS JOIN (VALUES (30), (50), (100), (200)) AS t(limiar)
GROUP BY t.limiar
ORDER BY t.limiar
"""

con.sql(q_limiares).df()

,limiar,countries_who_pass
0,30,58
1,50,46
2,100,31
3,200,17


In [40]:
q_cobertura = """
WITH por_pais AS (
  SELECT country,
         COUNT(*) FILTER (WHERE year = 2024) AS n_2024,
         COUNT(*) FILTER (WHERE year = 2025) AS n_2025
  FROM survey WHERE uses_ai IS NOT NULL GROUP BY country
)
SELECT t.limiar,
       COUNT(*) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar) AS paises,
       ROUND(100.0 * SUM(n_2024 + n_2025) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar)
             / SUM(n_2024 + n_2025), 1) AS pct_respostas_cobertas
FROM por_pais CROSS JOIN (VALUES (30), (50), (100), (200)) AS t(limiar)
GROUP BY t.limiar ORDER BY t.limiar
"""
con.sql(q_cobertura).df()

,limiar,paises,pct_respostas_cobertas
0,30,58,95.9
1,50,46,92.9
2,100,31,86.7
3,200,17,75.6


In [ ]:
MIN_N = 100

q_pais = f"""
WITH base AS (
  SELECT country, year,
         COUNT(uses_ai)    AS n,
         SUM(uses_ai::INT) AS n_usa
  FROM survey
  GROUP BY country, year
),
validos AS (
  SELECT country
  FROM base
  GROUP BY country
  HAVING COUNT(*) = 2 AND MIN(n) >= {MIN_N}
)
SELECT b.country, b.year, b.n, b.n_usa,
       ROUND(100.0 * b.n_usa / b.n, 1) AS pct_usa_ia
FROM base b
JOIN validos v USING (country)
ORDER BY b.country, b.year
"""
d_pais = con.sql(q_pais).df()
d_pais


,country,year,n,n_usa,pct_usa_ia
0,Australia,2024,466,262.0,56.2
1,Australia,2025,437,337.0,77.1
2,Austria,2024,310,200.0,64.5
3,Austria,2025,234,191.0,81.6
4,Belgium,2024,172,108.0,62.8
...,...,...,...,...,...
57,Ukraine,2025,430,399.0,92.8
58,United Kingdom of Great Britain and Northern I...,2024,1260,685.0,54.4
59,United Kingdom of Great Britain and Northern I...,2025,1223,879.0,71.9
60,United States of America,2024,4306,2432.0,56.5


In [42]:
nomes = {"United States of America": "United States",
         "United Kingdom of Great Britain and Northern Ireland": "United Kingdom"}
d = d_pais.copy()
d["country"] = d["country"].replace(nomes)

w = d.pivot(index="country", columns="year", values="pct_usa_ia").reset_index()
w = w.sort_values(2025)

fig = go.Figure()
for _, r in w.iterrows():
    fig.add_trace(go.Scatter(x=[r[2024], r[2025]], y=[r["country"]] * 2, mode="lines",
                             line=dict(color="lightgray", width=3),
                             showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=w[2024], y=w["country"], mode="markers", name="2024", marker=dict(size=10)))
fig.add_trace(go.Scatter(x=w[2025], y=w["country"], mode="markers", name="2025", marker=dict(size=10)))
fig.update_layout(title=f"Adoção de IA por país (n ≥ {MIN_N} por ano)",
                  xaxis_title="% that uses Ai", height=max(350, 22 * len(w)))
fig.show()

In [43]:
w = d.pivot(index="country", columns="year", values="pct_usa_ia").reset_index()
w["delta_pp"] = (w[2025] - w[2024]).round(1)
w = w.sort_values("delta_pp", ascending=False)

print(w[["country", 2024, 2025, "delta_pp"]].head(5))   # maiores subidas
print(w[["country", 2024, 2025, "delta_pp"]].tail(5))   # menores subidas

year      country  2024  2025  delta_pp
10         Greece  67.1  91.8      24.7
18    New Zealand  56.3  80.9      24.6
22        Romania  63.5  87.6      24.1
15          Italy  58.4  82.0      23.6
7         Finland  58.3  81.8      23.5
year       country  2024  2025  delta_pp
6          Denmark  63.2  77.1      13.9
27          Turkey  72.4  86.0      13.6
25          Sweden  68.0  80.1      12.1
19          Norway  67.1  78.7      11.6
23    South Africa  74.5  84.8      10.3


## Adoption by professional function

In [ ]:
q_function =f"""
WITH longo AS (
    SELECT year, uses_ai,
            TRIM(unnest(string_split(dev_type,';'))) AS function
    FROM survey
    WHERE dev_type Is NOT NULL
),
base AS (
    SELECT function, year,
            COUNT(uses_ai) AS n,
            SUM(uses_ai::INT)::INT AS n_usa
    FROM longo
    GROUP BY function, year
)
SELECT function, year, n, n_usa, ROUND(100.0 * n_usa / n, 1) as pct_use_ai
FROM base
WHERE function IN (SELECT function FROM base GROUP BY function HAVING COUNT(*) = 2 AND MIN(n) >= {MIN_N})
ORDER BY function, year
"""

d_func = con.sql(q_function).df()
d_func.head(6)

,function,year,n,n_usa,pct_use_ai
0,Academic researcher,2024,368,197,53.5
1,Academic researcher,2025,373,263,70.5
2,Cloud infrastructure engineer,2024,253,159,62.8
3,Cloud infrastructure engineer,2025,216,185,85.6
4,Data engineer,2024,509,306,60.1
5,Data engineer,2025,397,337,84.9
6,Data or business analyst,2024,182,111,61.0
7,Data or business analyst,2025,135,90,66.7
8,"Developer, QA or test",2024,216,132,61.1
9,"Developer, QA or test",2025,156,130,83.3
